# Raw vs Curated — human-reviewed test evaluation

This notebook evaluates **12 fixed checkpoints** (Raw/Curated × six architectures) using one unified evaluator on exactly the same held-out test:

```text
benchmakring_transmission_line_defect_model/data/finding_nemo_human_reviewed_6cls/
├── images/test   # 1,931 images
└── labels/test   # 1,931 label files, 2,850 boxes
```

This is **not** `val_clean`, `test_main_v1`, or the exploratory `finding_nemo_test_audit_v5_bundle.tar`. The v5 bundle is intentionally not used.

Expected hardware: RTX 3050/3060. Checkpoints run sequentially to fit limited VRAM. Existing per-run JSON files are reused, so the notebook can safely resume after interruption.


In [ ]:
# 1. Resolve repository and evaluation paths.
from pathlib import Path
import os, sys

def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'benchmakring_transmission_line_defect_model').is_dir():
            return p
    raise FileNotFoundError('Open this notebook from inside the yolov11 repository.')

REPO = find_repo(Path.cwd())
BENCH = REPO / 'benchmakring_transmission_line_defect_model'
DATASET_DIR = BENCH / 'data' / 'finding_nemo_human_reviewed_6cls'
DATASET_YAML = DATASET_DIR / 'data.yaml'
MODEL_ROOT = REPO / 'camber' / 'artifacts' / 'benchmark'
OUTPUT_DIR = BENCH / 'results_ablation' / 'finding_nemo' / 'raw_curated_human_reviewed_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repository :', REPO)
print('Dataset    :', DATASET_YAML)
print('Models     :', MODEL_ROOT)
print('Output     :', OUTPUT_DIR)


## Environment

Install CUDA-enabled PyTorch **before** running this notebook. The cell below deliberately does not reinstall PyTorch because doing so can replace a working CUDA build with a CPU build.


In [ ]:
# 2. Install evaluator dependencies. Restart the kernel if pip asks for it.
%pip install -q "ultralytics==8.4.172" "transformers==4.47.1" "accelerate>=1.0,<2" pycocotools pandas matplotlib pyyaml pillow


In [ ]:
# 3. Hardware fail-fast check and conservative automatic batches.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Install the CUDA build of PyTorch and restart the kernel; '
        'do not run the 12-model benchmark on CPU.'
    )

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 2**30
DEVICE = '0'
YOLO_BATCH = 16 if vram_gb >= 10 else (8 if vram_gb >= 6 else 4)
RTDETR_BATCH = 4 if vram_gb >= 10 else (2 if vram_gb >= 7 else 1)
print(f'GPU: {gpu.name} | VRAM: {vram_gb:.1f} GiB')
print(f'YOLO batch={YOLO_BATCH}; RT-DETR batch={RTDETR_BATCH}')


In [ ]:
# 4. Verify the exact human-reviewed test before spending GPU time.
image_dir = DATASET_DIR / 'images' / 'test'
label_dir = DATASET_DIR / 'labels' / 'test'
images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
labels = sorted(label_dir.glob('*.txt'))
boxes = sum(sum(bool(line.strip()) for line in p.read_text(encoding='utf-8', errors='ignore').splitlines()) for p in labels)
missing = [p.stem for p in images if not (label_dir / f'{p.stem}.txt').is_file()]

print(f'images={len(images):,}; labels={len(labels):,}; boxes={boxes:,}; missing_labels={len(missing)}')
assert DATASET_YAML.is_file()
assert len(images) == 1931, 'Expected exactly 1,931 human-reviewed test images.'
assert len(labels) == 1931 and not missing, 'Image/label pairing is not frozen as expected.'
assert boxes == 2850, 'Expected exactly 2,850 human-reviewed GT boxes.'
print('HUMAN-REVIEWED TEST VERIFIED')


In [ ]:
# 5. Fixed Raw/Curated checkpoint manifest.
CHECKPOINTS = {
    ('Raw','yolo11n'): MODEL_ROOT/'raw/yolo11n/weights/best.pt',
    ('Raw','yolo11m'): MODEL_ROOT/'raw/yolo11m/weights/best.pt',
    ('Raw','yolo26n'): MODEL_ROOT/'raw/yolo26n/weights/best.pt',
    ('Raw','yolo26m'): MODEL_ROOT/'raw/yolo26m/weights/best.pt',
    ('Raw','rtdetr50'): MODEL_ROOT/'raw/rtdetr50/best.pt',
    ('Raw','rtdetr101'): MODEL_ROOT/'raw/rtdetr101/best.pt',
    ('Curated','yolo11n'): MODEL_ROOT/'curated_latest/yolo11n/weights/best.pt',
    ('Curated','yolo11m'): MODEL_ROOT/'curated_latest/yolo11m/weights/best.pt',
    ('Curated','yolo26n'): MODEL_ROOT/'curated_latest/yolo26n/weights/best.pt',
    ('Curated','yolo26m'): MODEL_ROOT/'curated_latest/yolo26m/weights/best.pt',
    ('Curated','rtdetr50'): MODEL_ROOT/'curated_latest/r50vd/best.pt',
    ('Curated','rtdetr101'): MODEL_ROOT/'curated_latest/r101vd/best.pt',
}

STASH = {
    ('Raw','yolo11n'): 'stash://bunpmc/projects/raw_yolo11n/weights/best.pt',
    ('Raw','yolo11m'): 'stash://bunpmc/projects/raw_yolo11m/weights/best.pt',
    ('Raw','yolo26n'): 'stash://bunpmc/projects/raw_yolo26n/weights/best.pt',
    ('Raw','yolo26m'): 'stash://bunpmc/projects/raw_yolo26m/weights/best.pt',
    ('Raw','rtdetr50'): 'stash://bunpmc/projects/raw_rtdetr50/checkpoints/best.pt',
    ('Raw','rtdetr101'): 'stash://bunpmc/projects/raw_rtdetr101/checkpoints/best.pt',
    ('Curated','yolo11n'): 'stash://bunpmc/projects/finding_nemo/yolo11n/weights/best.pt',
    ('Curated','yolo11m'): 'stash://bunpmc/projects/finding_nemo/yolo11m/weights/best.pt',
    ('Curated','yolo26n'): 'stash://bunpmc/projects/finding_nemo/yolo26n/weights/best.pt',
    ('Curated','yolo26m'): 'stash://bunpmc/projects/finding_nemo/yolo26m/weights/best.pt',
    ('Curated','rtdetr50'): 'stash://bunpmc/projects/finding_nemo/rtdetr_training/finding_nemo_human_reviewed_6cls_20261101/r50vd/checkpoints/best.pt',
    ('Curated','rtdetr101'): 'stash://bunpmc/projects/finding_nemo/rtdetr_training/finding_nemo_human_reviewed_6cls_20261101/r101vd/checkpoints/best.pt',
}

FETCH_MISSING_FROM_CAMBER = True
missing_keys = [key for key, path in CHECKPOINTS.items() if not path.is_file()]
if missing_keys and FETCH_MISSING_FROM_CAMBER:
    import subprocess
    for key in missing_keys:
        destination = CHECKPOINTS[key]
        destination.parent.mkdir(parents=True, exist_ok=True)
        print(f'[CAMBER] {key} -> {destination.relative_to(REPO)}')
        subprocess.run(['camber','stash','cp',STASH[key],str(destination)], check=True)

missing_ckpt = [str(path) for path in CHECKPOINTS.values() if not path.is_file()]
if missing_ckpt:
    raise FileNotFoundError('Missing checkpoints after Camber restore:\n' + '\n'.join(missing_ckpt))

for (regime, model), path in CHECKPOINTS.items():
    print(f'{regime:7} {model:10} {path.stat().st_size/2**20:7.1f} MiB  {path.relative_to(REPO)}')


If the local model mirror is absent, restore the same paths using `camber stash cp`. The source prefixes are:

```text
Raw YOLO:       stash://bunpmc/projects/raw_<model>/weights/best.pt
Raw RT-DETR:    stash://bunpmc/projects/raw_rtdetr50|101/checkpoints/best.pt
Curated YOLO:   stash://bunpmc/projects/finding_nemo/<model>/weights/best.pt
Curated RT-DETR stash://bunpmc/projects/finding_nemo/rtdetr_training/finding_nemo_human_reviewed_6cls_20261101/<r50vd|r101vd>/checkpoints/best.pt
```


In [ ]:
# 6. Load the repository's unified evaluator.
import gc, importlib.util, json, time
import pandas as pd

module_path = BENCH / 'scripts' / 'evaluate_nemo_selected_checkpoints.py'
spec = importlib.util.spec_from_file_location('nemo_eval', module_path)
nemo_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nemo_eval)

MODEL_ORDER = ['yolo11n','yolo11m','yolo26n','yolo26m','rtdetr50','rtdetr101']
CLASSES = list(nemo_eval.CLASS_NAMES.values())
print('Unified evaluator loaded:', module_path)


In [ ]:
# 7. Evaluate sequentially. Re-run this cell to resume; completed JSON files are reused.
all_results = {}
for regime in ('Raw','Curated'):
    for model in MODEL_ORDER:
        checkpoint = CHECKPOINTS[(regime, model)]
        result_file = OUTPUT_DIR / f'{regime.lower()}_{model}.json'
        if result_file.is_file():
            result = json.loads(result_file.read_text(encoding='utf-8'))
            print(f'[REUSE] {regime:7} {model:10} mAP50={result["accuracy"]["mAP50"]:.4f}')
        else:
            print(f'\n[EVAL] {regime:7} {model:10} -> human-reviewed test')
            if model.startswith('yolo'):
                result = nemo_eval.evaluate_yolo_checkpoint(
                    model_key=model, checkpoint_path=checkpoint, dataset_yaml=DATASET_YAML,
                    split='test', device_str=DEVICE, batch_size=YOLO_BATCH, limit=None, imgsz=640)
            else:
                result = nemo_eval.evaluate_rtdetr_checkpoint(
                    model_key=model, checkpoint_path=checkpoint, dataset_yaml=DATASET_YAML,
                    split='test', device_str=DEVICE, batch_size=RTDETR_BATCH, limit=None, imgsz=640)
            result['regime'] = regime
            result['protocol'] = 'finding_nemo_human_reviewed_6cls/test (1931 images, 2850 boxes)'
            result_file.write_text(json.dumps(result, indent=2), encoding='utf-8')
            print(f'[SAVED] {result_file.name} | mAP50={result["accuracy"]["mAP50"]:.4f}')
        all_results[(regime, model)] = result
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
# 8. Aggregate matched overall and per-class CSVs.
overall_rows, class_rows = [], []
for (regime, model), result in all_results.items():
    a = result['accuracy']
    overall_rows.append({
        'Train': regime, 'Model': model, 'Precision': a['precision'], 'Recall': a['recall'],
        'F1': a['f1_score'], 'mAP50': a['mAP50'], 'mAP50:95': a['mAP50_95'],
        'MEAN_5': a['MEAN_5_defect_ap50'],
    })
    row = {'Train':regime, 'Model':model}
    for cls in CLASSES:
        row[f'{cls}_AP50'] = a['per_class'][cls]['ap50']
        row[f'{cls}_AP50_95'] = a['per_class'][cls]['ap50_95']
    class_rows.append(row)

overall = pd.DataFrame(overall_rows)
per_class = pd.DataFrame(class_rows)
overall.to_csv(OUTPUT_DIR/'benchmark_summary.csv', index=False)
per_class.to_csv(OUTPUT_DIR/'per_class_summary.csv', index=False)
display(overall)
print('Saved:', OUTPUT_DIR/'benchmark_summary.csv')
print('Saved:', OUTPUT_DIR/'per_class_summary.csv')


In [ ]:
# 9. One figure: Raw and Curated per-class performance on the SAME human-reviewed test.
import matplotlib.pyplot as plt
import numpy as np

labels = {'yolo11n':'YOLO11n','yolo11m':'YOLO11m','yolo26n':'YOLO26n','yolo26m':'YOLO26m',
          'rtdetr50':'RT-DETR-R50','rtdetr101':'RT-DETR-R101'}
class_labels = ['Insulator','Broken insulator','Flashover','Bird nest','Foreign object','Broken strand']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, (suffix, title) in enumerate((('AP50','AP@0.5 (%)'),('AP50_95','AP@0.5:0.95 (%)'))):
    mats = {}
    for regime in ('Raw','Curated'):
        d = per_class[per_class.Train == regime].set_index('Model').reindex(MODEL_ORDER)
        mats[regime] = d[[f'{c}_{suffix}' for c in CLASSES]].to_numpy(float) * 100
    lo, hi = min(x.min() for x in mats.values()), max(x.max() for x in mats.values())
    for row, regime in enumerate(('Raw','Curated')):
        ax = axes[row, col]
        m = mats[regime]
        im = ax.imshow(m, cmap='Blues', vmin=lo, vmax=hi, aspect='auto')
        for i in range(m.shape[0]):
            for j in range(m.shape[1]):
                ax.text(j, i, f'{m[i,j]:.1f}', ha='center', va='center',
                        color='white' if im.norm(m[i,j]) > .58 else 'black', fontsize=10)
        ax.set_xticks(range(6), class_labels, rotation=30, ha='right')
        ax.set_yticks(range(6), [labels[m] for m in MODEL_ORDER])
        ax.set_title(f'{regime} training — {title}', fontweight='bold')
        fig.colorbar(im, ax=ax, fraction=.035, pad=.02)

fig.suptitle('Per-class outcomes on the human-reviewed test (1,931 images, 2,850 boxes)', fontweight='bold')
fig.tight_layout()
png = OUTPUT_DIR/'per_class_raw_curated_human_reviewed_test.png'
pdf = OUTPUT_DIR/'per_class_raw_curated_human_reviewed_test.pdf'
fig.savefig(png, dpi=300, bbox_inches='tight')
fig.savefig(pdf, bbox_inches='tight')
plt.show()
print('Saved:', png)
print('Saved:', pdf)


## Reporting rule

Only the CSV and figure produced by this notebook may be described as a matched Raw-versus-Curated comparison on the human-reviewed test. Do not substitute validation results or operational-v5 exploratory results.
